# Nguyễn Thành Trí - MSSV: 6451071079
## Bài 2 (Trang 15-17) - Câu 7: Biến đổi các thuộc tính theo yêu cầu
1. Chuyển đổi thuộc tính giá vé (`fare`) về kiểu số nguyên.
2. Trích xuất thuộc tính danh hiệu (`title`) từ tên hành khách (`name`), phân loại các danh hiệu hiếm và mã hóa thành số (`Mr: 1, Miss: 2, Mrs: 3, Master: 4, Rare: 5`), sau đó xóa cột `name`.
3. Chuyển đổi thuộc tính giới tính (`sex`: male=0, female=1) và cổng lên tàu (`embarked`: S=0, C=1, Q=2) thành dạng số.
4. Chia thuộc tính tuổi (`age`) thành 8 nhóm cân bằng theo gợi ý.
5. Chia thuộc tính giá vé (`fare`) thành 6 nhóm cân bằng theo gợi ý.

In [1]:
# Nguyễn Thành Trí - 6451071079
# Bài 2 (15-17) - Câu 7: Biến đổi các thuộc tính

import os
import re
import numpy as np
import pandas as pd

print("Họ tên: Nguyễn Thành Trí - MSSV: 6451071079")

# --- NẠP DỮ LIỆU & ÁP DỤNG TIỀN XỬ LÝ TỪ CÂU TRƯỚC ---
def get_data_path(rel_path):
    candidates = [
        rel_path,
        os.path.join('..', rel_path),
        os.path.join('..', '..', rel_path),
        os.path.join('datasets', os.path.basename(rel_path)),
        os.path.join('..', 'datasets', os.path.basename(rel_path)),
        os.path.join('..', '..', 'datasets', os.path.basename(rel_path))
    ]
    for p in candidates:
        if os.path.exists(p):
            return p
    raise FileNotFoundError(f'Không tìm thấy file dữ liệu: {rel_path}')

csv_path = get_data_path('titanic.csv')
df = pd.read_csv(csv_path, na_values=['?'])
df.columns = df.columns.str.lower()
df['age'] = pd.to_numeric(df['age'], errors='coerce')
df['fare'] = pd.to_numeric(df['fare'], errors='coerce')

# Tiền xử lý Câu 6
deck_map = {"A": 1, "B": 2, "C": 3, "D": 4, "E": 5, "F": 6, "G": 7, "U": 8}
df['cabin'] = df['cabin'].fillna("U0")
df['deck'] = df['cabin'].map(lambda x: re.compile(r"([a-zA-Z]+)").search(x).group())
df['deck'] = df['deck'].map(deck_map).fillna(0).astype(int)
df = df.drop(['cabin', 'ticket', 'boat', 'body', 'home.dest'], axis=1, errors='ignore')

mean, std = df["age"].mean(), df["age"].std()
is_null = df["age"].isnull().sum()
np.random.seed(42)
rand_age = np.random.randint(mean - std, mean + std, size=is_null)
age_slice = df["age"].copy()
age_slice[np.isnan(age_slice)] = rand_age
df["age"] = age_slice.astype(int)
df['embarked'] = df['embarked'].fillna('S')
df['fare'] = df['fare'].fillna(0)

# --- THỰC HIỆN CÁC BIẾN ĐỔI THEO YÊU CẦU CÂU 7 ---

# 1. Chuyển đổi giá vé (fare) về kiểu nguyên
df['fare'] = df['fare'].astype(int)

# 2. Trích xuất danh hiệu (title) từ tên và mã hóa thành số
titles = {"Mr": 1, "Miss": 2, "Mrs": 3, "Master": 4, "Rare": 5}
df['title'] = df['name'].str.extract(r' ([A-Za-z]+)\.', expand=False)
df['title'] = df['title'].replace(['Lady', 'Countess', 'Capt', 'Col', 'Don', 'Dr',
                                   'Major', 'Rev', 'Sir', 'Jonkheer', 'Dona'], 'Rare')
df['title'] = df['title'].replace('Mlle', 'Miss')
df['title'] = df['title'].replace('Ms', 'Miss')
df['title'] = df['title'].replace('Mme', 'Mrs')
df['title'] = df['title'].map(titles).fillna(0).astype(int)
df = df.drop(['name'], axis=1)

# 3. Chuyển đổi giới tính (sex) và cổng lên tàu (embarked) thành số
genders = {"male": 0, "female": 1}
df['sex'] = df['sex'].map(genders)

ports = {"S": 0, "C": 1, "Q": 2}
df['embarked'] = df['embarked'].map(ports)

# 4. Chia giá trị thuộc tính tuổi (age) thành 8 nhóm
df.loc[df['age'] <= 11, 'age'] = 0
df.loc[(df['age'] > 11) & (df['age'] <= 18), 'age'] = 1
df.loc[(df['age'] > 18) & (df['age'] <= 22), 'age'] = 2
df.loc[(df['age'] > 22) & (df['age'] <= 27), 'age'] = 3
df.loc[(df['age'] > 27) & (df['age'] <= 33), 'age'] = 4
df.loc[(df['age'] > 33) & (df['age'] <= 40), 'age'] = 5
df.loc[(df['age'] > 40) & (df['age'] <= 66), 'age'] = 6
df.loc[df['age'] > 66, 'age'] = 7

# 5. Chia giá trị thuộc tính giá vé (fare) thành 6 nhóm
df.loc[df['fare'] <= 7.91, 'fare'] = 0
df.loc[(df['fare'] > 7.91) & (df['fare'] <= 14.454), 'fare'] = 1
df.loc[(df['fare'] > 14.454) & (df['fare'] <= 31), 'fare'] = 2
df.loc[(df['fare'] > 31) & (df['fare'] <= 99), 'fare'] = 3
df.loc[(df['fare'] > 99) & (df['fare'] <= 250), 'fare'] = 4
df.loc[df['fare'] > 250, 'fare'] = 5
df['fare'] = df['fare'].astype(int)

print("--- PHÂN BỐ CÁC NHÓM TUỔI (age: 0 đến 7) ---")
print(df['age'].value_counts().sort_index())

print("\n--- PHÂN BỐ CÁC NHÓM GIÁ VÉ (fare: 0 đến 5) ---")
print(df['fare'].value_counts().sort_index())

Họ tên: Nguyễn Thành Trí - MSSV: 6451071079
--- PHÂN BỐ CÁC NHÓM TUỔI (age: 0 đến 7) ---
age
0     91
1    144
2    176
3    214
4    224
5    202
6    249
7      9
Name: count, dtype: int64

--- PHÂN BỐ CÁC NHÓM GIÁ VÉ (fare: 0 đến 5) ---
fare
0    361
1    312
2    325
3    227
4     67
5     17
Name: count, dtype: int64
